# Study CAX M1 misalignment effects on the X-ray beam

```2026/07/10```

In [ ]:
import os
import sys

sys.path.append(os.path.expanduser("~/repos/cax-scripts"))
# sys.path.append(os.path.expanduser("~/repos/cax-control"))

from shadowpy.cax_simulation import CAXSim
import numpy as np
import matplotlib.pyplot as plt
from shadowpy.utils import save_image

In [ ]:
cax = CAXSim(total_rays=500000)


In [ ]:
cax.dvf_B1.image.img.shape

In [ ]:
cax.mirror.tilt = np.array([0.005, 0.004, 0.03])

In [ ]:
cax.mirror.tilt

In [ ]:
cax.mirror.shadow_oe.OFFZ

In [ ]:
cax.dvf_A1.shadow_oe.RZ_SLIT[0]

In [ ]:
cax.mirror.tx = -1

In [ ]:
(0.1-0.28)/2

In [ ]:
cax.dvf_B1.image.img.shape

In [ ]:
(cax.dvf_B1.image.x_bin_centers[1] - cax.dvf_B1.image.x_bin_centers[0])*1000

In [ ]:
beamr = cax.dvf_B1.beam.duplicate()
# beamr.retrace(12641) 
ana = save_image(cax.dvf_B1, beamr)

In [ ]:
image_ticket = cax.dvf_B1.beam.histo2(col_h=3, col_v=1, 
                            nbins_h=320, nbins_v=240, nolost=1)

In [ ]:
hist = np.array(image_ticket['histogram'])
h_centers = np.array(image_ticket['bin_h_center'])
v_centers = np.array(image_ticket['bin_v_center'])

In [ ]:
cx = h_centers[np.argmax(np.sum(hist, axis=1))]*1000
cy = v_centers[np.argmax(np.sum(hist, axis=0))]*1000

In [ ]:
cy

In [ ]:
cx

In [ ]:
beamr.histo2()

In [ ]:
# cax.mirror.ry = 1000*np.deg2rad(1e-3)
cax.mirror.offset = [0, 0, 0]


In [ ]:
ana.print_stats(ana.hprm_fitting)

In [ ]:
cax.dvf_B1.image.plot(hprm=cax.dvf_B1.image.hprm_momenta)

In [ ]:
from optlnls.plot import plot_beam


In [ ]:
x = h_centers
y = v_centers
img = hist

beam2D_2 = np.zeros((len(y)+1, len(x)+1))
beam2D_2[0,1:] = x
beam2D_2[1:,0] = y
beam2D_2[1:,1:] = img.T


plot_beam(beam2D_2, cut=0, textA=1, textB=10, textC=9, textD=0, fitType=1,
          plot_title='CAX M1 nominal', show_colorbar=1,
            x_range=1, y_range=1, x_range_min=-0.12, x_range_max=0.12, y_range_min=-0.12, y_range_max=0.12,
          zero_pad_x=1, zero_pad_y=1, cmap='viridis', integral=1e11,
          units=2)


In [ ]:
x = cax.dvf_B1.image.x_bin_centers
y = cax.dvf_B1.image.y_bin_centers
img = cax.dvf_B1.image.img

beam2D = np.zeros((len(y)+1, len(x)+1))
beam2D[0,1:] = x
beam2D[1:,0] = y
beam2D[1:,1:] = img.T


plot_beam(beam2D, cut=0, textA=1, textB=10, textC=9, textD=0, fitType=1,
          plot_title='CAX M1 nominal', show_colorbar=1,
            x_range=1, y_range=1, x_range_min=-0.12, x_range_max=0.12, y_range_min=-0.12, y_range_max=0.12,
          zero_pad_x=1, zero_pad_y=1, cmap='viridis', integral=1e11,
          units=2)


In [ ]:
cax.dvf_B1.source_distance

In [ ]:
cax.dvf_B1.image.fit(hprm=cax.dvf_B1.image.hprm_momenta)

In [ ]:
print(cax.dvf_B1.image.hprm_fitting['mux']*1000)
print(cax.dvf_B1.image.hprm_fitting['muy']*1000)

In [ ]:
cax.dvf_B1.image.print_stats(cax.dvf_B1.image.hprm_fitting)

In [ ]:
cax.dvf_B1.image.img.shape

In [ ]:
cx, cy = cax.dvf_B1.image._centroid_from_projection()
droi = 4

In [ ]:
roi_avg = np.mean(
    cax.dvf_B1.image.img[cx - droi : cx + droi,
    cy - droi : cy + droi]
    )
mean = np.mean(cax.dvf_B1.image.img)
ratio = roi_avg / mean if mean != 0 else 0

print(roi_avg, mean, ratio)

In [ ]:
def xp2(x):
    x += 2
    return x
x = 2
print(x)
x = xp2(x)
print(x)


In [ ]:
def _perform_plot_caustic(cax: CAXSim):
    dist, fwhm_x, fwhm_y = \
        cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                    (-250, 250),
                                    n_points=100, 
                                    max_workers=10)

    astig = dist[np.argmin(fwhm_x)] - dist[np.argmin(fwhm_y)]

    plt.figure(figsize=(12, 6))
    plt.plot(dist, 1000*fwhm_x, marker='o', markersize=5, label="FWHM X")
    plt.plot(dist, 1000*fwhm_y, marker='o', markersize=5, label="FWHM Y")
    plt.vlines(dist[np.argmin(fwhm_x)], 0, 1000*np.min(fwhm_x), color='blue', linestyle='--', label=f"Min FWHM X: {1000*np.min(fwhm_x):.2f} μm at {dist[np.argmin(fwhm_x)]:.2f} mm")
    plt.vlines(dist[np.argmin(fwhm_y)], 0, 1000*np.min(fwhm_y), color='orange', linestyle='--', label=f"Min FWHM Y: {1000*np.min(fwhm_y):.2f} μm at {dist[np.argmin(fwhm_y)]:.2f} mm")
    plt.xlabel('Distance from screen (mm)')
    plt.ylabel('FWHM (μm)')
    plt.title(f'Caustic of the beam: astigmatism of {astig:.2f} mm')
    plt.legend()
    plt.grid()
    plt.show()

    return astig

In [ ]:
_perform_plot_caustic(cax)

In [ ]:
def _astigmatism_acquisition(cax: CAXSim):

    # Initial search

    low_lim = -1000
    hgh_lim =  1000

    done = False

    while not done:
        distances, fwhm_x, fwhm_y = \
            cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                        s_range=(low_lim, hgh_lim),
                                        n_points=50, 
                                        max_workers=10)
        # prior focal lengths
        fx = distances[np.nanargmin(fwhm_x)]
        fy = distances[np.nanargmin(fwhm_y)]

        print(fx, fy)

        done = True

        # check if the search was enought otherwise repeat
        if low_lim in [fx, fy]:
            low_lim -= 100
            done = False

        if hgh_lim in [fx, fy]:
            hgh_lim += 100
            done = False

    # Finer search
    distances, fwhm_x, fwhm_y = \
    cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                s_range=(np.min([fx, fy])-50, np.max([fx, fy])+50),
                                n_points=100, 
                                max_workers=10)

    # actual focal lengths
    fx = distances[np.argmin(fwhm_x)]
    fy = distances[np.argmin(fwhm_y)]
    print(fx, fy)

    f = np.mean([fx, fy])
    retr_beam = cax.dvf_B1.beam.duplicate()
    retr_beam.retrace(f)
    ana = save_image(cax.dvf_B1, retr_beam)
    astig = fy - fx

    return distances, fwhm_x, fwhm_y, fx, fy, f, astig, ana 

In [ ]:
def acquisition_func(cax: CAXSim, tx_vals: np.ndarray, ry_vals: np.ndarray):
    

In [ ]:
distances, fwhm_x, fwhm_y, fx, fy, f, astig, ana = _astigmatism_acquisition(cax)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(distances, 1000*fwhm_x, marker='o', markersize=5, label="FWHM X")
plt.plot(distances, 1000*fwhm_y, marker='o', markersize=5, label="FWHM Y")
plt.vlines(distances[np.argmin(fwhm_x)], 0, 1000*np.min(fwhm_x), color='blue', linestyle='--', label=f"Min FWHM X: {1000*np.min(fwhm_x):.2f} μm at {distances[np.argmin(fwhm_x)]:.2f} mm")
plt.vlines(distances[np.argmin(fwhm_y)], 0, 1000*np.min(fwhm_y), color='orange', linestyle='--', label=f"Min FWHM Y: {1000*np.min(fwhm_y):.2f} μm at {distances[np.argmin(fwhm_y)]:.2f} mm")
plt.vlines(f, 0, 50, color='black', label=f"Midpoint of the foci: {f:.2f} mm")
plt.xlabel('Distance from screen (mm)')
plt.ylabel('FWHM (μm)')
plt.title(f'Caustic of the beam: astigmatism of {astig:.2f} mm')
plt.legend()
plt.grid()
plt.show()

In [ ]:
np.deg2rad(0.01)

In [ ]:
cax.mirror.ry

In [ ]:
cax.mirror.ry = 0.01

In [ ]:
ana.print_stats(ana.hprm_fitting)

In [ ]:
# cax.mirror.ry = -11.e-4
# cax.mirror.ry
# cax.mirror.tx = 10.e-3
# cax.mirror.rx = 2.e-3

cax.mirror.tx = 0.1
cax.mirror.ry = 0.01

In [ ]:
_perform_plot_caustic(cax)

In [ ]:
plt.imshow(cax.mirror.image.img.T)

In [ ]:
cax.dvf_B1.image.plot()

In [ ]:
cax.dvf_B1.image.print_stats(hprm=cax.dvf_B1.image.hprm_fitting)

In [ ]:
cax.dvf_B1.image.print_stats(hprm=cax.dvf_B1.image.hprm_fitting)

In [ ]:
cax.mirror.tx
np.rad2deg(cax.mirror.rx)

In [ ]:
cax.mirror.offset

In [ ]:
cax.mirror.rx = 0.1

In [ ]:
cax.mirror.shadow_oe.Z_ROT

In [ ]:
np.deg2rad(0.1)

In [ ]:
cax.mirror.tilt

In [ ]:
cax.mirror.image.plot()

In [ ]:
def CAX_SVD(cax: CAXSim, dT: float = 0.01, dR: float = 0.01):
    """
    Perform SVD analysis on the CAX simulation to determine the sensitivity of the system to misalignments.

    Parameters:
    cax (CAXSim): The CAX simulation object.
    dT (float): The translation step size in mm.
    dR (float): The rotation step size in degrees.

    Returns:
    None
    """
    
    T0 = cax.mirror.offset
    R0 = cax.mirror.tilt

    observables = ['fwhmx', 'fwhmy', 'peak', 'astigmatism']
    knobs = ['tx', 'ty', 'rx', 'ry', 'rz']

    fwhm_x0 = cax.dvf_B1.image.hprm_fitting['fwhmx']
    fwhm_y0 = cax.dvf_B1.image.hprm_fitting['fwhmy']
    peak0 = cax.dvf_B1.image.img.max()

    caustic_offsets = np.linspace(-150, 100, 100)

    dist, fwhm_x, fwhm_y = \
        cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                    (-250, 250),
                                    n_points=100, 
                                    max_workers=10)

    astigmatism0 = dist[np.argmin(fwhm_x)] - dist[np.argmin(fwhm_y)]

    # Jacobian matrix
    J = np.zeros((len(observables), 5))  # 5 misalignment parameters: tx, ty, rx, ry, rz

    for i, obs in enumerate(observables):
        # Perturb each misalignment parameter and compute the change in the observable
        for j, param in enumerate(knobs):
            # Save original value
            original_value = getattr(cax.mirror, param)

            # Perturb the parameter
            if param in ['tx', 'ty']:
                setattr(cax.mirror, param, original_value + dT)
                delta = dT
            else:
                setattr(cax.mirror, param, np.rad2deg(original_value) + dR)
                delta = dR

            # Compute the observable after perturbation
            if obs == 'fwhmx':
                J[i, j] = (cax.dvf_B1.image.hprm_fitting['fwhmx'] - fwhm_x0)/delta
            elif obs == 'fwhmy':
                J[i, j] = (cax.dvf_B1.image.hprm_fitting['fwhmy'] - fwhm_y0)/delta
            elif obs == 'peak':
                J[i, j] = (cax.dvf_B1.image.img.max() - peak0)/delta
            elif obs == 'astigmatism':
                dist, fwhm_x, fwhm_y = \
                    cax.dvf_B1.parallel_caustic(cax.dvf_B1.beam,
                                    (-250, 250),
                                    n_points=100, 
                                    max_workers=10)

                astig = dist[np.argmin(fwhm_x)] - dist[np.argmin(fwhm_y)]
                J[i, j] = (astig - astigmatism0)/delta
            # Restore original value
            setattr(cax.mirror, param, original_value)

    print(f"Jacobian matrix: {J}")

    U, S, Vt = np.linalg.svd(J, full_matrices=False)

    return J, U, S, Vt


In [ ]:
cax = CAXSim(total_rays=500000)

In [ ]:
J, U, S, Vt = CAX_SVD(cax, dT=0.01, dR=0.01)

In [ ]:
# jacobian

for i, obs in enumerate(['fwhmx', 'fwhmy', 'peak', 'astigmatism']):
    print(f"Observable: {obs}")
    for j, param in enumerate(['tx', 'ty', 'rx', 'ry', 'rz']):
        print(f"  Sensitivity to {param}: {J[i, j]:.4f}")

In [ ]:
# Mode 0:

print(f"Singular value associated with mode 0: {S[0]}")

print(f"Combination of knobs to turn: {np.round(Vt[0], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 0: {np.round(U.T[0], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
print("Conclusion: turning RY has a strong, isolated effect on astigmatism")
print("=="*40)


# Mode 1:

print(f"Singular value associated with mode 1: {S[1]}")

print(f"Combination of knobs to turn: {np.round(Vt[1], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 1: {np.round(U.T[1], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
print("Conclusion: turning RX decreases the measured peak")
print("=="*40)


# Mode 2:

print(f"Singular value associated with mode 2: {S[2]}")

print(f"Combination of knobs to turn: {np.round(Vt[2], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 2: {np.round(U.T[2], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
# print("Conclusion: turning RY has a strong, isolated effect on astigmatism")
print("=="*40)

# Mode 0:

print(f"Singular value associated with mode 3: {S[3]}")

print(f"Combination of knobs to turn: {np.round(Vt[3], 3)}")
print("*Order of knobs: tx, ty, rx, ry, rz")

print(f"Observable effect by exciting mode 3: {np.round(U.T[3], 3)}")
print("*Order of observables: fwhm_x, fwhm_y, peak, astigmatism")

print("=="*40)
print("Conclusion: turning RY has a strong, isolated effect on astigmatism")
print("=="*40)









In [ ]:
desired_observables = np.array([46.85, 27.08, 150, 0])

In [ ]:
J_inv = Vt.T @ np.diag(1/S) @ U.T

In [ ]:
J_inv @ desired_observables

In [ ]:
cax.dvf_B1.image.plot()